In [1]:
UNI_RANDOM_SEED = 2024

import numpy as np
import torch
import torch.nn.functional as F

np.random.seed(UNI_RANDOM_SEED) 
torch.manual_seed(UNI_RANDOM_SEED)

torch.cuda.manual_seed(UNI_RANDOM_SEED)
torch.cuda.manual_seed_all(UNI_RANDOM_SEED)

import pdb
from pathlib import Path

try:
    import open3d
    from visual_utils import open3d_vis_utils as V
    OPEN3D_FLAG = True
except:
    import mayavi.mlab as mlab
    from visual_utils import visualize_utils as V
    OPEN3D_FLAG = False

from pcdet.datasets.kitti.kitti_dataset import create_kitti_infos
from pcdet.config import cfg, cfg_from_yaml_file
from pcdet.datasets import KittiDataset, build_dataloader
from pcdet.models import build_network, load_data_to_gpu
from pcdet.utils import common_utils

from data_tools import adv_dataset, kitti_carla_dataset
from eval_utils import eval_utils


EVAL_OUTPUT_DIR = "./eval_output/"
CFG_FILE = "./cfgs/kitti_models/pointrcnn.yaml"
DATA_CONFIG_FILE = "./cfgs/dataset_configs/kitti_dataset.yaml"
DATA_PATH = "/home/ksas/Public/datasets/KITTI"
CKPT_PATH = "/home/ksas/Public/model_zoo/pcdet/pointrcnn_7870.pth"

BATCH_SIZE = 1
WORKERS = 4
DIST_TEST = False

cfg_from_yaml_file(CFG_FILE, cfg)

# BATCH_SIZE = cfg.OPTIMIZATION.BATCH_SIZE_PER_GPU
logger = common_utils.create_logger()
logger.info('-----------------Kitti Attack Test-------------------------')


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
[Open3D INFO] Resetting default logger to print to terminal.


/home/ksas/chw_space/OpenPCDet_Developing/pcdet/models/detectors/__init__.py:20: UserWarning: You are using a variant OpenPCDet modified by uzuki-dev, NOT AN ORIGINAL VERSION!
  warnings.warn("You are using a variant OpenPCDet modified by uzuki-dev, NOT AN ORIGINAL VERSION!")
2024-01-12 15:36:45,753   INFO  -----------------Kitti Attack Test-------------------------


In [2]:
test_set, test_loader, sampler = build_dataloader(
        dataset_cfg=cfg.DATA_CONFIG,
        class_names=cfg.CLASS_NAMES,
        batch_size=BATCH_SIZE,
        dist=DIST_TEST, workers=WORKERS, logger=logger, training=False
    )
logger.info(f'Class names of samples: \t{test_set.class_names}')

2024-01-12 15:36:45,771   INFO  Loading KITTI dataset
2024-01-12 15:36:45,860   INFO  Total samples for KITTI dataset: 3769
2024-01-12 15:36:45,861   INFO  Class names of samples: 	['Car', 'Pedestrian', 'Cyclist']


In [3]:
carla_adv_dataset = adv_dataset(test_set)

2024-01-12 15:36:45,874   INFO  Total samples for dataset: 3769


In [4]:
model = build_network(model_cfg=cfg.MODEL, num_class=len(cfg.CLASS_NAMES), dataset=test_set)
model.load_params_from_file(filename=CKPT_PATH, logger=logger, to_cpu=True)
model.cuda()
model.eval()

for idx, module in enumerate(model.module_list):
    logger.info(f'Module names of model \t({idx}): \t{module._get_name()}')
    
backbone_network = model.module_list[0]
point_headbox = model.module_list[1]
pointrcnn_head = model.module_list[2]

2024-01-12 15:36:46,694   INFO  ==> Loading parameters from checkpoint /home/ksas/Public/model_zoo/pcdet/pointrcnn_7870.pth to CPU
2024-01-12 15:36:46,716   INFO  ==> Done (loaded 309/309)
2024-01-12 15:36:46,726   INFO  Module names of model 	(0): 	PointNet2MSG
2024-01-12 15:36:46,727   INFO  Module names of model 	(1): 	PointHeadBox
2024-01-12 15:36:46,727   INFO  Module names of model 	(2): 	PointRCNNHead


In [5]:
for i, batch_dict in enumerate(carla_adv_dataset):
        load_data_to_gpu(batch_dict)
        logger.info(f"keys of batch dict: \t{batch_dict.keys()}")
        
        model.eval()
        model.pseudo_train()
        model.zero_grad()
        pred_dicts, _ = model(batch_dict)
        # pred_dicts, _, _ = model(batch_dict)

        
        loss, tb_dict, disp_dict = model.get_training_loss()
        logger.info(f"total loss: \t{loss}")
        
        loss_dict = {}
       
        point_headbox_cls_loss, cls_loss_dict = point_headbox.get_cls_layer_loss()
        point_headbox_box_loss, box_loss_dict = point_headbox.get_box_layer_loss()
        loss_dict.update(cls_loss_dict)
        loss_dict.update(box_loss_dict)
        
        rcnn_cls_loss, cls_loss_dict = pointrcnn_head.get_box_cls_layer_loss()
        rcnn_reg_loss, reg_loss_dict = pointrcnn_head.get_box_reg_layer_loss()
        loss_dict.update(cls_loss_dict)
        loss_dict.update(reg_loss_dict)
        
        logger.info(f"loss dict: \t{loss_dict}")
        
        V.draw_scenes(
            points=batch_dict['points'][:, 1:], ref_boxes=pred_dicts[0]['pred_boxes'].detach(),
            ref_scores=pred_dicts[0]['pred_scores'].detach(), ref_labels=pred_dicts[0]['pred_labels'].detach(), gt_boxes=batch_dict['gt_boxes'][0]
        )
        
        import pdb
        pdb.set_trace()

2024-01-12 15:36:46,759   INFO  keys of batch dict: 	dict_keys(['frame_id', 'calib', 'gt_boxes', 'points', 'lidar_aug_matrix', 'use_lead_xyz', 'image_shape', 'batch_size'])


2024-01-12 15:36:47,661   INFO  total loss: 	1.5699479579925537
2024-01-12 15:36:47,666   INFO  loss dict: 	{'point_loss_cls': 0.832926869392395, 'point_pos_num': 27.0, 'point_loss_box': 0.43472909927368164, 'rcnn_loss_cls': 0.06797345727682114, 'rcnn_loss_reg': 0.19643718004226685, 'rcnn_loss_corner': 0.0378812775015831}


[Open3D WARNING] invalid color in PaintUniformColor, clipping to [0, 1]
[Open3D WARNING] invalid color in PaintUniformColor, clipping to [0, 1]
> /tmp/ipykernel_4039382/4213558702.py(1)<module>()
----> 1 for i, batch_dict in enumerate(carla_adv_dataset):
      2         load_data_to_gpu(batch_dict)
      3         logger.info(f"keys of batch dict: \t{batch_dict.keys()}")
      4 
      5         model.eval()



In [6]:
eval_utils.eval_one_epoch(
        cfg, None, model, carla_adv_dataset, 0, logger, dist_test=DIST_TEST,
        result_dir=Path(EVAL_OUTPUT_DIR)
        , infer_time=True
    )

2024-01-12 15:36:58,518   INFO  *************** EPOCH 0 EVALUATION *****************
eval:   0%|          | 0/3769 [00:00<?, ?it/s]

eval: 100%|██████████| 3769/3769 [04:18<00:00, 14.55it/s, infer_time=52.52(50.72), recall_0.3=(15668, 15690) / 17558]
2024-01-12 15:41:17,476   INFO  *************** Performance of EPOCH 0 *****************
2024-01-12 15:41:17,477   INFO  Generate label finished(sec_per_example: 0.0000 second).
2024-01-12 15:41:17,477   INFO  recall_roi_0.3: 0.892357
2024-01-12 15:41:17,478   INFO  recall_rcnn_0.3: 0.893610
2024-01-12 15:41:17,478   INFO  recall_roi_0.5: 0.828682
2024-01-12 15:41:17,478   INFO  recall_rcnn_0.5: 0.842978
2024-01-12 15:41:17,478   INFO  recall_roi_0.7: 0.535710
2024-01-12 15:41:17,479   INFO  recall_rcnn_0.7: 0.664540
2024-01-12 15:41:17,483   INFO  Average predicted number of objects(3769 samples): 4.646
/home/ksas/chw_space/OpenPCDet_Developing/pcdet/datasets/kitti/kitti_object_eval_python/eval.py:10: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, b

{'recall/roi_0.3': 0.8923567604510765,
 'recall/rcnn_0.3': 0.893609750541064,
 'recall/roi_0.5': 0.8286820822417131,
 'recall/rcnn_0.5': 0.8429775600865702,
 'recall/roi_0.7': 0.5357102175646429,
 'recall/rcnn_0.7': 0.6645403804533546,
 'Car_aos/easy_R40': 90.60927422678226,
 'Car_aos/moderate_R40': 84.52182868018238,
 'Car_aos/hard_R40': 81.52117540938806,
 'Car_3d/easy_R40': 78.19494448665908,
 'Car_3d/moderate_R40': 67.90899156066665,
 'Car_3d/hard_R40': 65.35699888855375,
 'Car_bev/easy_R40': 88.36398685939399,
 'Car_bev/moderate_R40': 82.99075382280378,
 'Car_bev/hard_R40': 80.85842818623811,
 'Car_image/easy_R40': 91.75229284546107,
 'Car_image/moderate_R40': 86.88871513653041,
 'Car_image/hard_R40': 84.59820968835223,
 'Pedestrian_aos/easy_R40': 49.0353865432687,
 'Pedestrian_aos/moderate_R40': 41.873160746145096,
 'Pedestrian_aos/hard_R40': 35.64959501933734,
 'Pedestrian_3d/easy_R40': 43.838949659700596,
 'Pedestrian_3d/moderate_R40': 37.22973227289476,
 'Pedestrian_3d/hard_R4

2024-01-12 15:41:37,291   INFO  
Car AP@0.70, 0.70, 0.70:
bbox AP:88.5812, 86.7496, 80.1062
bev  AP:87.0447, 79.0308, 78.6136
3d   AP:75.1943, 67.0116, 65.7010
aos  AP:87.57, 84.41, 77.59
Car AP_R40@0.70, 0.70, 0.70:
bbox AP:91.7523, 86.8887, 84.5982
bev  AP:88.3640, 82.9908, 80.8584
3d   AP:78.1949, 67.9090, 65.3570
aos  AP:90.61, 84.52, 81.52
Car AP@0.70, 0.50, 0.50:
bbox AP:88.5812, 86.7496, 80.1062
bev  AP:89.3926, 88.3620, 88.0661
3d   AP:89.2757, 87.9698, 87.8601
aos  AP:87.57, 84.41, 77.59
Car AP_R40@0.70, 0.50, 0.50:
bbox AP:91.7523, 86.8887, 84.5982
bev  AP:92.6839, 90.0779, 87.8932
3d   AP:92.5467, 89.8120, 87.6309
aos  AP:90.61, 84.52, 81.52
Pedestrian AP@0.50, 0.50, 0.50:
bbox AP:55.4177, 47.0196, 40.1301
bev  AP:47.6578, 44.3817, 37.6502
3d   AP:44.6757, 37.4365, 35.0575
aos  AP:51.77, 44.09, 37.67
Pedestrian AP_R40@0.50, 0.50, 0.50:
bbox AP:52.6747, 45.0703, 38.4328
bev  AP:47.9480, 40.6975, 34.4781
3d   AP:43.8389, 37.2297, 31.1593
aos  AP:49.04, 41.87, 35.65
Pedestrian AP@0.50, 0.25, 0.25:
bbox AP:55.4177, 47.0196, 40.1301
bev  AP:56.7015, 48.1542, 40.5216
3d   AP:56.6573, 48.0890, 40.4630
aos  AP:51.77, 44.09, 37.67
Pedestrian AP_R40@0.50, 0.25, 0.25:
bbox AP:52.6747, 45.0703, 38.4328
bev  AP:56.8266, 47.3270, 40.3676
3d   AP:56.7731, 47.2649, 40.2969
aos  AP:49.04, 41.87, 35.65
Cyclist AP@0.50, 0.50, 0.50:
bbox AP:61.5252, 43.0417, 35.0103
bev  AP:60.3704, 41.4714, 34.0138
3d   AP:59.9614, 33.9537, 33.7593
aos  AP:59.96, 41.67, 34.21
Cyclist AP_R40@0.50, 0.50, 0.50:
bbox AP:60.2571, 37.6600, 35.3160
bev  AP:58.9961, 36.0958, 33.8039
3d   AP:56.3758, 33.7256, 31.4339
aos  AP:58.55, 36.37, 34.18
Cyclist AP@0.50, 0.25, 0.25:
bbox AP:61.5252, 43.0417, 35.0103
bev  AP:60.6254, 41.8412, 34.3412
3d   AP:60.6254, 41.8412, 34.3412
aos  AP:59.96, 41.67, 34.21
Cyclist AP_R40@0.50, 0.25, 0.25:
bbox AP:60.2571, 37.6600, 35.3160
bev  AP:59.2935, 36.5666, 34.3006
3d   AP:59.2935, 36.5666, 34.3006
aos  AP:58.55, 36.37, 34.18

In [7]:
eval_utils.eval_one_epoch(
        cfg, None, model, test_loader, 0, logger, dist_test=DIST_TEST,
        result_dir=Path(EVAL_OUTPUT_DIR)
        , infer_time=True
    )

2024-01-12 15:44:17,571   INFO  *************** EPOCH 0 EVALUATION *****************
eval: 100%|██████████| 3769/3769 [03:08<00:00, 19.97it/s, infer_time=47.52(47.07), recall_0.3=(16938, 16973) / 17558]
2024-01-12 15:47:26,349   INFO  *************** Performance of EPOCH 0 *****************
2024-01-12 15:47:26,351   INFO  Generate label finished(sec_per_example: 0.0001 second).
2024-01-12 15:47:26,352   INFO  recall_roi_0.3: 0.964688
2024-01-12 15:47:26,353   INFO  recall_rcnn_0.3: 0.966682
2024-01-12 15:47:26,354   INFO  recall_roi_0.5: 0.919125
2024-01-12 15:47:26,355   INFO  recall_rcnn_0.5: 0.929035
2024-01-12 15:47:26,355   INFO  recall_roi_0.7: 0.682994
2024-01-12 15:47:26,356   INFO  recall_rcnn_0.7: 0.762672
2024-01-12 15:47:26,360   INFO  Average predicted number of objects(3769 samples): 5.551
/home/ksas/miniconda3/envs/pcdet_develop/lib/python3.9/site-packages/numba/cuda/dispatcher.py:536: NumbaPerformanceWarning: Grid size 12 will likely result in GPU under-utilization due 

{'recall/roi_0.3': 0.9646884611003531,
 'recall/rcnn_0.3': 0.9666818544253332,
 'recall/roi_0.5': 0.9191251851008088,
 'recall/rcnn_0.5': 0.9290351976307096,
 'recall/roi_0.7': 0.6829935072331701,
 'recall/rcnn_0.7': 0.7626722861373733,
 'Car_aos/easy_R40': 99.05567834195024,
 'Car_aos/moderate_R40': 93.89847156401363,
 'Car_aos/hard_R40': 93.74270451650503,
 'Car_3d/easy_R40': 92.18300809440063,
 'Car_3d/moderate_R40': 81.41084572642201,
 'Car_3d/hard_R40': 80.88722209576522,
 'Car_bev/easy_R40': 95.90498235977266,
 'Car_bev/moderate_R40': 90.33323315116974,
 'Car_bev/hard_R40': 90.22023107446604,
 'Car_image/easy_R40': 99.08872482902001,
 'Car_image/moderate_R40': 94.02994053716847,
 'Car_image/hard_R40': 93.95938714087985,
 'Pedestrian_aos/easy_R40': 80.71908639919151,
 'Pedestrian_aos/moderate_R40': 76.4246795711534,
 'Pedestrian_aos/hard_R40': 71.59872183638021,
 'Pedestrian_3d/easy_R40': 71.83587032033351,
 'Pedestrian_3d/moderate_R40': 65.97590667007516,
 'Pedestrian_3d/hard_R40

2024-01-12 15:47:35,915   INFO  
Car AP@0.70, 0.70, 0.70:
bbox AP:98.0106, 90.4862, 90.3078
bev  AP:90.3662, 88.9447, 88.5946
3d   AP:89.2901, 79.2153, 78.7551
aos  AP:97.96, 90.39, 90.13
Car AP_R40@0.70, 0.70, 0.70:
bbox AP:99.0887, 94.0299, 93.9594
bev  AP:95.9050, 90.3332, 90.2202
3d   AP:92.1830, 81.4108, 80.8872
aos  AP:99.06, 93.90, 93.74
Car AP@0.70, 0.50, 0.50:
bbox AP:98.0106, 90.4862, 90.3078
bev  AP:98.3867, 90.6892, 90.6100
3d   AP:98.2945, 90.6742, 90.5853
aos  AP:97.96, 90.39, 90.13
Car AP_R40@0.70, 0.50, 0.50:
bbox AP:99.0887, 94.0299, 93.9594
bev  AP:99.3582, 96.7091, 96.7086
3d   AP:99.3153, 96.6458, 96.5752
aos  AP:99.06, 93.90, 93.74
Pedestrian AP@0.50, 0.50, 0.50:
bbox AP:82.6938, 80.3032, 74.3339
bev  AP:72.9366, 69.3558, 63.9712
3d   AP:71.2096, 64.4160, 61.5648
aos  AP:79.80, 76.41, 70.46
Pedestrian AP_R40@0.50, 0.50, 0.50:
bbox AP:83.7870, 80.5743, 75.8810
bev  AP:74.0078, 69.2262, 65.4119
3d   AP:71.8359, 65.9759, 60.7654
aos  AP:80.72, 76.42, 71.60
Pedestrian AP@0.50, 0.25, 0.25:
bbox AP:82.6938, 80.3032, 74.3339
bev  AP:88.3206, 86.8871, 84.5226
3d   AP:88.3137, 86.8690, 84.3266
aos  AP:79.80, 76.41, 70.46
Pedestrian AP_R40@0.50, 0.25, 0.25:
bbox AP:83.7870, 80.5743, 75.8810
bev  AP:92.8533, 91.0422, 86.0141
3d   AP:92.8371, 90.8833, 85.9042
aos  AP:80.72, 76.42, 71.60
Cyclist AP@0.50, 0.50, 0.50:
bbox AP:97.0807, 79.1073, 78.1093
bev  AP:96.3020, 77.9603, 76.5960
3d   AP:88.6547, 75.9801, 68.9831
aos  AP:96.98, 78.31, 77.23
Cyclist AP_R40@0.50, 0.50, 0.50:
bbox AP:97.8278, 83.0833, 78.6728
bev  AP:97.3316, 81.5356, 77.1819
3d   AP:94.0496, 76.4288, 71.8527
aos  AP:97.72, 82.14, 77.80
Cyclist AP@0.50, 0.25, 0.25:
bbox AP:97.0807, 79.1073, 78.1093
bev  AP:96.5096, 78.0618, 76.8287
3d   AP:96.5096, 78.0618, 76.8287
aos  AP:96.98, 78.31, 77.23
Cyclist AP_R40@0.50, 0.25, 0.25:
bbox AP:97.8278, 83.0833, 78.6728
bev  AP:97.4958, 81.7290, 77.3664
3d   AP:97.4958, 81.7290, 77.3664
aos  AP:97.72, 82.14, 77.80